# Complementos de Investigação Operacional — capítulo 4: Teoria da substituição
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/applied-operations-research-course/blob/main/code/python/notebooks/04_replacement.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 4 of the course "Complements of Operational Research" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/applied-operations-research-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("applied-operations-research-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/applied-operations-research-course
    RAIZ = os.path.abspath(os.path.join("applied-operations-research-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 4.1 Teoria da substituição: equipamentos que se desgastam

Exemplo [`ex04_1_wear.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/04_replacement/04_1_wear/ex04_1_wear.py) · funções da UC: [`desgaste.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/04_replacement/04_1_wear/desgaste.py)

```text
Reproduz os exemplos do deck 4.1 (equipamentos que se desgastam, sem valor temporal do dinheiro).

Exemplo-guia: carrinha a gasóleo, P = 30000, C_n = 3000, ..., 12500, S_n = 22000, ..., 5000.
Custo médio anual mínimo A(5) = 8760: substituir ao fim de 5 anos; fundo achatado (A(4) e A(6) a cerca de 1%).
Regra: M_5 = 8400 < A(4) = 8850 (manter), M_6 = 9100 > A(5) = 8760 (substituir).
Duas máquinas: carrinha elétrica, P = 44000, mínimo 8316,67 aos 6 anos; a gasóleo com 3 anos troca-se
ao fim do 4.º ano (M_4 = 7900 < 8316,67, M_5 = 8400 > 8316,67). Código do slide «Em Python»: 5 8760.0.
Para resolver na aula: Exemplo 1 (7 anos, 1821,43), Exemplo 2 (6 anos, 6333,33) e
Exemplo 3 (duas máquinas: trocar A pela B no fim do ano 2 ou do ano 3; comprar agora a B).
```

In [2]:
import numpy as np

from desgaste import custo_medio, custo_mais_um_ano, media, melhor, quando_trocar


def perto(a, b, tol=1e-9):
    """Verdadeiro se todos os valores de a estão a menos de tol dos de b (tolerância absoluta)."""
    return bool(np.max(np.abs(np.asarray(a, float) - np.asarray(b, float))) <= tol)


def fmt(v, f="%.2f"):
    """Valores separados por espaços, todos com o mesmo formato."""
    return " ".join(f % x for x in v)


def tabela(L):
    print("     n     C_n   soma C   P - S_n    total       A(n)")
    for n, c, soma, perda, tot, a in L:
        print("  %4d %7d %8d %9d %8d %10.2f" % (n, c, soma, perda, tot, a))


def mostra_troca(linhas, minimo):
    """Imprime a comparação de quando_trocar, ano a ano."""
    sinais = {"manter": "<", "indiferente": "=", "trocar": ">"}
    for k, M, dec in linhas:
        print("  %d.º ano da antiga: M = %d %s %.2f -> %s" % (k, M, sinais[dec], minimo, dec))


ok = True
print("Complementos de IO — deck 4.1: equipamentos que se desgastam")

# ------------------------------------------------------------ carrinha a gasóleo
P = 30000
C = [3000, 3600, 4400, 5400, 6400, 7600, 9800, 12500]
S = [22000, 17000, 13500, 11000, 9000, 7500, 6200, 5000]
print("\nCarrinha a gasóleo (P = 30000)")
L = media(P, C, S)
tabela(L)
b = melhor(L)
print("  substituir ao fim de %d anos: A = %.2f € por ano" % (b[0], b[5]))
perda = [(P - S[n - 1]) / n for n in range(1, 9)]
func = [L[n - 1][2] / n for n in range(1, 9)]
print("  perda de valor por ano: %s" % fmt(perda, "%.1f"))
print("  funcionamento por ano:  %s" % fmt(func, "%.1f"))
print("  fundo achatado: A(4) %+.2f%% e A(6) %+.2f%% em relação a A(5)"
      % (100 * (L[3][5] / L[4][5] - 1), 100 * (L[5][5] / L[4][5] - 1)))
for n in (4, 5):
    M = custo_mais_um_ano(C, S, n)
    print("  regra: M_%d = %d + %d = %d %s A(%d) = %.2f -> %s"
          % (n + 1, C[n], S[n - 1] - S[n], M, "<" if M < L[n - 1][5] else ">", n, L[n - 1][5],
             "manter" if M < L[n - 1][5] else "substituir"))
ok &= (b[0] == 5 and abs(b[5] - 8760) < 1e-9
       and perto([r[5] for r in L], [11000, 9800, 27500 / 3, 8850, 8760, 52900 / 6, 64000 / 7, 9712.5])
       and perto(perda, [8000, 6500, 5500, 4750, 4200, 3750, 3400, 3125])
       and abs(func[2] - 3666.67) < 0.01 and abs(func[6] - 5742.86) < 0.01
       and custo_mais_um_ano(C, S, 4) == 8400 and custo_mais_um_ano(C, S, 5) == 9100)

# ------------------------------------------------------------ duas máquinas: a elétrica
Pe = 44000
Ce = [1600, 1900, 2400, 3100, 4000, 5400, 7000, 9000]
Se = [32000, 25500, 21000, 17500, 14800, 12500, 10600, 9000]
Le = media(Pe, Ce, Se)
be = melhor(Le)
print("\nCarrinha elétrica (P = 44000)")
print("  A(n): %s" % fmt([r[5] for r in Le]))
print("  mínimo: %.2f € por ano, substituindo ao fim de %d anos" % (be[5], be[0]))
print("Gasóleo com 3 anos: trocar já pela elétrica?")
quando, linhas = quando_trocar(C, S, 3, be[5])
mostra_troca(linhas, be[5])
print("  -> manter a gasóleo e trocar ao fim do %d.º ano" % quando[0])
ok &= (be[0] == 6 and abs(be[5] - 8316.67) < 0.005 and quando == [4]
       and perto([r[5] for r in Le], [13600, 11000, 9633, 8875, 8440, 8317, 8400, 8675], 0.5))

# ------------------------------------------------------------ código do slide «Em Python»
A = custo_medio(P, C, S)
n = A.index(min(A)) + 1
print("\nComo no slide «Em Python»: %d %.1f" % (n, A[n - 1]))
ok &= n == 5 and A[n - 1] == 8760.0

# ------------------------------------------------------------ Para resolver na aula: Exemplos 1 e 2
for nome, Px, Cx, Sx, n_ok, a_ok in (
        ("Exemplo 1 (P = 7100, sucata 100)", 7100, [200, 350, 500, 700, 1000, 1300, 1700, 2100], 100, 7, 1821.43),
        ("Exemplo 2 (P = 24400, sucata 400)", 24400, [400, 1000, 1600, 2400, 3600, 5000, 6400, 8000], 400, 6, 6333.33)):
    Lx = media(Px, Cx, Sx)
    bx = melhor(Lx)
    print("\n%s" % nome)
    print("  A(n): %s" % fmt([r[5] for r in Lx]))
    print("  substituir ao fim de %d anos: A = %.2f" % (bx[0], bx[5]))
    ok &= bx[0] == n_ok and abs(bx[5] - a_ok) < 0.005

# ------------------------------------------------------------ Exemplo 3: duas máquinas, sem revenda
CA = [2000 + 10000 * k for k in range(8)]
CB = [3000 + 4000 * k for k in range(8)]
AA, AB = custo_medio(45000, CA), custo_medio(55000, CB)
bA, bB = melhor(media(45000, CA)), melhor(media(55000, CB))
print("\nExemplo 3 (duas máquinas, sem revenda)")
print("  A: A(n) = %s; mínimo %.2f aos %d anos" % (fmt(AA), bA[5], bA[0]))
print("  B: A(n) = %s; mínimo %.2f aos %d anos" % (fmt(AB), bB[5], bB[0]))
print("a) A em funcionamento (no 1.º ano); mais um ano de A custa C_{n+1}:")
q3, linhas = quando_trocar(CA, 0, 1, bB[5])
mostra_troca(linhas, bB[5])
print("  -> trocar A pela B no fim do ano %s" % " ou do ano ".join(str(k) for k in q3))
print("b) comprar agora: %s (%.2f contra %.2f)" % ("B" if bB[5] < bA[5] else "A", bB[5], bA[5]))
ok &= (bA[0] == 3 and bA[5] == 27000 and bB[0] == 5 and bB[5] == 22000 and q3 == [2, 3]
       and perto(AA[:5], [47000, 29500, 27000, 28250, 31000])
       and perto(AB[:7], [58000, 32500, 76000 / 3, 22750, 22000, 133000 / 6, 160000 / 7]))

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 4.1: equipamentos que se desgastam

Carrinha a gasóleo (P = 30000)
     n     C_n   soma C   P - S_n    total       A(n)
     1    3000     3000      8000    11000   11000.00
     2    3600     6600     13000    19600    9800.00
     3    4400    11000     16500    27500    9166.67
     4    5400    16400     19000    35400    8850.00
     5    6400    22800     21000    43800    8760.00
     6    7600    30400     22500    52900    8816.67
     7    9800    40200     23800    64000    9142.86
     8   12500    52700     25000    77700    9712.50
  substituir ao fim de 5 anos: A = 8760.00 € por ano
  perda de valor por ano: 8000.0 6500.0 5500.0 4750.0 4200.0 3750.0 3400.0 3125.0
  funcionamento por ano:  3000.0 3300.0 3666.7 4100.0 4560.0 5066.7 5742.9 6587.5
  fundo achatado: A(4) +1.03% e A(6) +0.65% em relação a A(5)
  regra: M_5 = 6400 + 2000 = 8400 < A(4) = 8850.00 -> manter
  regra: M_6 = 7600 + 1500 = 9100 > A(5) = 8760.00 -> substituir

Carrinha elétri

## 4.2 Teoria da substituição: com valor temporal do dinheiro

Exemplo [`ex04_2_discounting.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/04_replacement/04_2_discounting/ex04_2_discounting.py) · funções da UC: [`valor_temporal.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/04_replacement/04_2_discounting/valor_temporal.py), [`desgaste.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/04_replacement/04_1_wear/desgaste.py)

```text
Reproduz os exemplos do deck 4.2 (substituição com valor temporal do dinheiro).

Fator de desconto d = 1/(1+r); com r = 10%: d = 0,9091 e 1000 € daqui a 5 anos valem hoje 620,92 €.
Carrinha a gasóleo de 4.1 com r = 10%: W(n) mínimo 10191,0 aos 6 anos (sem desconto eram 5 anos).
Regra: M_6 = 9781,8 < W(5) = 10251,9 (manter), M_7 = 11663,6 > W(6) = 10191,0 (substituir).
Ótimo e taxa (código do slide «Em Python»): 0 5 8760.0 | 0.05 6 9514.6 | 0.1 6 10191.0 | 0.15 6 10846.0.
Para resolver na aula: Exemplo 4 (P = 500, r = 5%): 3 anos, W = 271,61; sem desconto também 3 anos (266,67).
```

In [3]:
import numpy as np

from valor_temporal import custo_equivalente, desconto
from desgaste import custo_medio, custo_mais_um_ano, melhor


def perto(a, b, tol=1e-9):
    """Verdadeiro se todos os valores de a estão a menos de tol dos de b (tolerância absoluta)."""
    return bool(np.max(np.abs(np.asarray(a, float) - np.asarray(b, float))) <= tol)


def fmt(v, f="%.2f"):
    """Valores separados por espaços, todos com o mesmo formato."""
    return " ".join(f % x for x in v)


ok = True
print("Complementos de IO — deck 4.2: com valor temporal do dinheiro")

# ------------------------------------------------------------ fator de desconto
d = 1 / 1.1
print("\nFator de desconto com r = 10%%: d = %.4f; 1000 € daqui a 5 anos valem hoje %.2f €" % (d, 1000 * d ** 5))
ok &= abs(1000 * d ** 5 - 620.92) < 0.005

# ------------------------------------------------------------ carrinha com r = 10%
P = 30000
C = [3000, 3600, 4400, 5400, 6400, 7600, 9800, 12500]
S = [22000, 17000, 13500, 11000, 9000, 7500, 6200, 5000]
W = desconto(P, C, 0.10, S)
print("\nCarrinha a gasóleo com r = 10% (P = 30000)")
print("    n     C_n  d^(n-1)  C_n d^(n-1)  soma C d   S_n d^n  P+soma-S d^n  soma d      W(n)")
for n, c, f, cf, E, num, H, w in W:
    print("  %3d %7d %8.4f %12.1f %9.1f %9.1f %13.1f %7.4f %9.1f" % (n, c, f, cf, E, P + E - num, num, H, w))
b = melhor(W)
print("  substituir ao fim de %d anos: W = %.1f € por ano (sem desconto: 5 anos)" % (b[0], b[7]))
for n in (5, 6):
    M = custo_mais_um_ano(C, S, n, d)
    print("  regra: M_%d = %d + %d - %.4f x %d = %.1f %s W(%d) = %.1f -> %s"
          % (n + 1, C[n], S[n - 1], d, S[n], M, "<" if M < W[n - 1][7] else ">", n, W[n - 1][7],
             "manter" if M < W[n - 1][7] else "substituir"))
ok &= (b[0] == 6 and abs(b[7] - 10191.0) < 0.05
       and perto([r[7] for r in W], [13000, 11640.7, 10881.4, 10454.4, 10251.9, 10191.0, 10346.2, 10679.2], 0.05)
       and perto([P + r[4] - r[5] for r in W], [20000, 14049.6, 10142.7, 7513.1, 5588.3, 4233.6, 3181.6, 2332.5], 0.05)
       and perto([r[6] for r in W], [1, 1.9091, 2.7355, 3.4869, 4.1699, 4.7908, 5.3553, 5.8684], 5e-5)
       and abs(custo_mais_um_ano(C, S, 5, d) - 9781.8) < 0.05 and abs(custo_mais_um_ano(C, S, 6, d) - 11663.6) < 0.05)

# ------------------------------------------------------------ ótimo e taxa (código do slide «Em Python»)
print("\nÓtimo e taxa de desconto (como no slide «Em Python»):")
res = []
for r in (0, 0.05, 0.10, 0.15):
    Wr = custo_equivalente(P, C, S, r)
    n = Wr.index(min(Wr)) + 1
    res.append("%g %d %.1f" % (r, n, min(Wr)))
    ok &= abs(min(Wr) - {0: 8760.0, 0.05: 9514.6, 0.10: 10191.0, 0.15: 10846.0}[r]) < 0.05 and n == (5 if r == 0 else 6)
print("  " + " | ".join(res))

# ------------------------------------------------------------ Para resolver na aula: Exemplo 4
C4 = [0, 100, 200, 300, 400]
W4 = desconto(500, C4, 0.05)
print("\nExemplo 4 (P = 500, r = 5%, sucata 0)")
print("    n  C_n d^(n-1)  soma C d  soma d    W(n)")
for n, c, f, cf, E, num, H, w in W4:
    print("  %3d %11.2f %9.2f %7.4f %7.2f" % (n, cf, E, H, w))
b4 = melhor(W4)
print("  substituir ao fim de %d anos: W = %.2f" % (b4[0], b4[7]))
print("  regra (sem revenda): C_3 = %d <= W(2) = %.2f e C_4 = %d > W(3) = %.2f" % (C4[2], W4[1][7], C4[3], W4[2][7]))
A4 = custo_medio(500, C4)
print("  sem desconto: A(n) = %s -> %d anos" % (fmt(A4), A4.index(min(A4)) + 1))
ok &= (b4[0] == 3 and abs(b4[7] - 271.61) < 0.005
       and perto([r[7] for r in W4], [500, 304.88, 271.61, 278.20, 300.24], 0.005)
       and C4[2] <= W4[1][7] and C4[3] > W4[2][7]
       and A4.index(min(A4)) == 2 and perto(A4, [500, 300, 800 / 3, 275, 300]))

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 4.2: com valor temporal do dinheiro

Fator de desconto com r = 10%: d = 0.9091; 1000 € daqui a 5 anos valem hoje 620.92 €

Carrinha a gasóleo com r = 10% (P = 30000)
    n     C_n  d^(n-1)  C_n d^(n-1)  soma C d   S_n d^n  P+soma-S d^n  soma d      W(n)
    1    3000   1.0000       3000.0    3000.0   20000.0       13000.0  1.0000   13000.0
    2    3600   0.9091       3272.7    6272.7   14049.6       22223.1  1.9091   11640.7
    3    4400   0.8264       3636.4    9909.1   10142.7       29766.3  2.7355   10881.4
    4    5400   0.7513       4057.1   13966.2    7513.1       36453.0  3.4869   10454.4
    5    6400   0.6830       4371.3   18337.5    5588.3       42749.2  4.1699   10251.9
    6    7600   0.6209       4719.0   23056.5    4233.6       48822.9  4.7908   10191.0
    7    9800   0.5645       5531.8   28588.3    3181.6       55406.7  5.3553   10346.2
    8   12500   0.5132       6414.5   35002.8    2332.5       62670.3  5.8684   10679.2
  substituir ao 

## 4.3 Teoria da substituição: falhas súbitas e substituição de grupo

Exemplo [`ex04_3_group_replacement.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/04_replacement/04_3_group_replacement/ex04_3_group_replacement.py) · funções da UC: [`grupo.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/04_replacement/04_3_group_replacement/grupo.py), [`desgaste.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/04_replacement/04_1_wear/desgaste.py)

```text
Reproduz os exemplos do deck 4.3 (falhas súbitas e substituição de grupo).

Exemplo-guia: 800 luminárias, c_i = 25 €, c_g = 6 € por luminária, P(t) = 0,03 0,08 0,20 0,45 0,75 1 (semestres).
Vida média 4,49 semestres; individual: 800/4,49 = 178,2 falhas e K_ind = 4454,34 € por semestre.
N_t = 24; 40,72; 98,42; 207,87; 262,04; 247,45; 88,29; ...; grupo de 3 em 3 semestres: K(3) = 2959,5 €
(poupança de 1494,8 € por semestre, -34%). Leitura marginal: 25 N_4 = 5197 > K(3), 25 N_3 = 2461 < K(2).
Sensibilidade: c_g = 10, K(3) = 4026 (grupo); c_g = 12, K(3) = 4560 > 4454 (individual).
Simulação de Monte Carlo (4000 corridas, semente 1) dos N_t, comparada com a fórmula com tolerância 1,0.
Para resolver na aula: Exemplo 5 (1000 lâmpadas, c_i = 3, c_g = 1): vida 3,45 semanas, K_ind = 869,57,
grupo de 4 em 4 semanas, K(4) = 863,57 (compensa, por pouco).
```

In [4]:
import numpy as np

from grupo import falhas, grupo, simula
from desgaste import melhor


def perto(a, b, tol=1e-9):
    """Verdadeiro se todos os valores de a estão a menos de tol dos de b (tolerância absoluta)."""
    return bool(np.max(np.abs(np.asarray(a, float) - np.asarray(b, float))) <= tol)


def fmt(v, f="%.2f"):
    """Valores separados por espaços, todos com o mesmo formato."""
    return " ".join(f % x for x in v)


ok = True
print("Complementos de IO — deck 4.3: falhas súbitas e substituição de grupo")

# ------------------------------------------------------------ luminárias
Pac = [0.03, 0.08, 0.20, 0.45, 0.75, 1.0]
n, ci, cg = 800, 25, 6
p, vida, ind, G = grupo(n, ci, cg, Pac)
_, _, N = falhas(n, Pac)                       # N_1, ..., N_12
print("\nLuminárias (n = 800, c_i = 25 €, c_g = 6 € por luminária; períodos de um semestre)")
print("  p_t: %s" % fmt(p))
print("  vida média: %.2f semestres" % vida)
print("  individual: n/vida = %.1f falhas por semestre; K_ind = %.2f € por semestre" % (n / vida, ind))
print("  N_t (t = 1, ..., 12): %s" % fmt(N))
print("  política de grupo de t em t semestres:")
print("     t     N_t   soma N  25 soma N    total     K(t)")
for t, Nt, soma, cs, ncg, tot, K in G:
    print("  %4d %7.2f %8.2f %10.1f %8.1f %8.1f" % (t, Nt, soma, cs, tot, K))
b = melhor(G)
print("  melhor: grupo de %d em %d semestres, K = %.1f € por semestre, contra %.1f da individual" % (b[0], b[0], b[6], ind))
print("  poupança: %.1f € por semestre (%.0f%%), %.1f € por ano"
      % (ind - b[6], 100 * (b[6] / ind - 1), 2 * (ind - b[6])))
print("  leitura marginal: 25 x N_3 = %.1f < K(2) = %.1f; 25 x N_4 = %.1f > K(3) = %.1f"
      % (ci * N[2], G[1][6], ci * N[3], G[2][6]))
ok &= (perto(p, [0.03, 0.05, 0.12, 0.25, 0.30, 0.25], 1e-12) and abs(vida - 4.49) < 1e-9
       and abs(n / vida - 178.2) < 0.05 and abs(ind - 4454.34) < 0.005
       and perto(N, [24, 40.72, 98.42, 207.87, 262.04, 247.45, 88.29, 138.14, 190.73, 215.67, 194.40, 162.39],
                       0.005)
       # o slide faz a tabela com os N_t já arredondados (K(6) = 4468,8; sem arredondar, 4468,75):
       and perto([r[6] for r in G], [5400.0, 3209.0, 2959.5, 3518.8, 4125.3, 4468.8], 0.06)
       and perto([r[5] for r in G], [5400.0, 6418.0, 8878.5, 14075.3, 20626.3, 26812.5], 0.06)
       and b[0] == 3 and abs(ind - b[6] - 1494.8) < 0.05 and round(100 * (b[6] / ind - 1)) == -34
       and ci * N[3] > G[2][6] and ci * N[2] < G[1][6])

# ------------------------------------------------------------ sensibilidade a c_g
for cgx, K3_ok in ((10, 4026.2), (12, 4559.5)):
    _, _, _, Gx = grupo(n, ci, cgx, Pac)
    K3 = Gx[2][6]
    print("  sensibilidade: c_g = %d: K(3) = %.1f %s %.1f -> %s"
          % (cgx, K3, "<" if K3 < ind else ">", ind, "grupo" if K3 < ind else "individual"))
    ok &= abs(K3 - K3_ok) < 0.05 and melhor(Gx)[0] == 3

# ------------------------------------------------------------ simulação
T = 8
F = simula(n, Pac, T=T, runs=4000, seed=1)
print("\nSimulação de Monte Carlo (4000 corridas de 800 luminárias, semente 1)")
print("     t  N_t (fórmula)  simulação")
for t in range(T):
    print("  %4d %14.1f %10.1f" % (t + 1, N[t], F[t]))
print("  maior diferença: %s" % ("até 1,0 (dentro da tolerância)" if np.max(np.abs(F - N[:T])) <= 1.0 else "mais de 1,0"))
ok &= np.max(np.abs(F - np.array(N[:T]))) <= 1.0

# ------------------------------------------------------------ Para resolver na aula: Exemplo 5
p5, v5, i5, G5 = grupo(1000, 3, 1, [0.10, 0.25, 0.50, 0.70, 1.00])
b5 = melhor(G5)
print("\nExemplo 5 (1000 lâmpadas, c_i = 3 €, c_g = 1 €; períodos de uma semana)")
print("  p_t: %s" % fmt(p5))
print("  a) vida média: %.2f semanas" % v5)
print("  b) individual: K_ind = %.2f € por semana" % i5)
print("  c) N_t: %s" % fmt([r[1] for r in G5]))
print("     K(t): %s" % fmt([r[6] for r in G5]))
print("     melhor: grupo de %d em %d semanas, K = %.2f %s %.2f -> %s"
      % (b5[0], b5[0], b5[6], "<" if b5[6] < i5 else ">", i5, "compensa" if b5[6] < i5 else "não compensa"))
ok &= (perto(p5, [0.10, 0.15, 0.25, 0.20, 0.30], 1e-12) and abs(v5 - 3.45) < 1e-9 and abs(i5 - 869.57) < 0.005
       and perto([r[1] for r in G5], [100, 160, 281, 277.1, 429.86], 0.005)
       and perto([r[6] for r in G5], [1300, 890, 874.33, 863.57, 948.78], 0.006)
       and b5[0] == 4 and b5[6] < i5)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 4.3: falhas súbitas e substituição de grupo

Luminárias (n = 800, c_i = 25 €, c_g = 6 € por luminária; períodos de um semestre)
  p_t: 0.03 0.05 0.12 0.25 0.30 0.25
  vida média: 4.49 semestres
  individual: n/vida = 178.2 falhas por semestre; K_ind = 4454.34 € por semestre
  N_t (t = 1, ..., 12): 24.00 40.72 98.42 207.87 262.04 247.45 88.29 138.14 190.73 215.67 194.40 162.39
  política de grupo de t em t semestres:
     t     N_t   soma N  25 soma N    total     K(t)
     1   24.00    24.00      600.0   5400.0   5400.0
     2   40.72    64.72     1618.0   6418.0   3209.0
     3   98.42   163.14     4078.5   8878.5   2959.5
     4  207.87   371.01     9275.3  14075.3   3518.8
     5  262.04   633.05    15826.3  20626.3   4125.3
     6  247.45   880.50    22012.5  26812.5   4468.7
  melhor: grupo de 3 em 3 semestres, K = 2959.5 € por semestre, contra 4454.3 da individual
  poupança: 1494.8 € por semestre (-34%), 2989.7 € por ano
  leitura marginal: 25 x N_3 = 2


Simulação de Monte Carlo (4000 corridas de 800 luminárias, semente 1)
     t  N_t (fórmula)  simulação
     1           24.0       24.0
     2           40.7       40.7
     3           98.4       98.5
     4          207.9      207.7
     5          262.0      262.2
     6          247.4      247.3
     7           88.3       88.4
     8          138.1      138.3
  maior diferença: até 1,0 (dentro da tolerância)

Exemplo 5 (1000 lâmpadas, c_i = 3 €, c_g = 1 €; períodos de uma semana)
  p_t: 0.10 0.15 0.25 0.20 0.30
  a) vida média: 3.45 semanas
  b) individual: K_ind = 869.57 € por semana
  c) N_t: 100.00 160.00 281.00 277.10 429.86
     K(t): 1300.00 890.00 874.33 863.57 948.78
     melhor: grupo de 4 em 4 semanas, K = 863.57 < 869.57 -> compensa

confere com os slides: sim
